# ADTA-DAST 5770 — Group 4 — Q&A Search System
## Notebook 1 — PHASES 1 and 2 only

**Phases covered:**
- Phase 1: Set up development environment, installations, imports, GCP auth, Vertex AI init
- Phase 2: Process documents — list and load 100 PDFs from GCS bucket

Members: Karan Parekh (Group Leader), Sanjana Pendyala Ravinder, Sana Mhapsekar, Medina Maloku  
Domain: Sustainable Supply Chain Management  
Project: precise-machine-471801-n5  
Region: us-central1


## PHASE 1: SET UP DEVELOPMENT ENVIRONMENT — INSTALLATIONS — IMPORTS


### 1.1 Installations


In [1]:
# Core GCP / Vertex AI SDKs
%pip install --upgrade google-cloud-aiplatform --quiet
%pip install --upgrade vertexai --quiet
%pip install --upgrade google-genai --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-aiplatform[agent-engines]<2.0.0,>=1.132.0, but you have google-cloud-aiplatform 1.71.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 19.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [2]:
# LangChain + all required sub-packages
%pip install --upgrade langchain langchain-core langchain-classic langchain-community --quiet
%pip install --upgrade langchain-text-splitters --quiet
%pip install --upgrade langchain-google-community langchain-google-vertexai langchain-google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
google-adk 1.29.0 requires google-cloud-aiplatform[agent-engines]<2.0.0,

In [3]:
# PDF + OCR dependencies for unstructured document loading
!sudo apt -y -qq install tesseract-ocr libtesseract-dev
!sudo apt-get -y -qq install poppler-utils
%pip install --user --upgrade unstructured pdf2image pytesseract pdfminer.six unstructured_pytesseract --quiet
%pip install --user --upgrade pillow-heif opencv-python unstructured-inference pikepdf pypdf pi_heif --quiet

tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following additional packages will be installed:
  libarchive-dev libleptonica-dev
The following NEW packages will be installed:
  libarchive-dev libleptonica-dev libtesseract-dev
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 3,743 kB of archives.
After this operation, 16.0 MB of additional disk space will be used.
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/liba/libarchive/libarchive-dev_3.6.0-1ubuntu1.5_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a 

### 1.2 Authenticate with GCP and set project


In [1]:
import sys
from google.colab import auth
from google.cloud import storage

# 1. Authenticate when running inside Google Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()

# 2. Set project ID (Group 4's GCP project)
PROJECT_ID = 'precise-machine-471801-n5'
!gcloud config set project {PROJECT_ID}

# 3. Initialize storage client
storage_client = storage.Client(project=PROJECT_ID)
print(f"Authenticated with project: {storage_client.project}")

# Verify host project
!gcloud config get-value project

Updated property [core/project].
Authenticated with project: precise-machine-471801-n5
precise-machine-471801-n5


### 1.3 Imports from GCP Vertex AI


In [2]:
# ---- IMPORT FROM GCP: VERTEX AI ----
from google.cloud import aiplatform
import vertexai

# Namespace and NumericNamespace are used later for filtered vector-search retrieval
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import (
    Namespace,
    NumericNamespace,
)

/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### 1.4 Imports from LangChain (modern package structure)


In [3]:
# NEW: chains live in langchain_classic (the package that now hosts all chain code)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# NEW:
from langchain_google_community import GCSDirectoryLoader, GCSFileLoader

# NEW:
from langchain_core.prompts import PromptTemplate

# CHANGE: ChatPromptTemplate is imported from langchain_core.prompts because
# create_stuff_documents_chain (the replacement for RetrievalQA) requires a
# ChatPromptTemplate rather than a plain PromptTemplate.
from langchain_core.prompts import ChatPromptTemplate

# NEW (single correct import):
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CHANGE: PyPDFLoader import from langchain_community.document_loaders is correct
# and remains unchanged. This is the proper location in the reorganized ecosystem.
from langchain_community.document_loaders import PyPDFLoader

In [4]:
# ---- IMPORT FROM GCP: VERTEX AI & LANGCHAIN API ----
# CHANGE: These imports from langchain_google_vertexai remain correct and unchanged.
# - VertexAI: LLM wrapper for Vertex AI text models (e.g., text-bison, gemini)
# - VertexAIEmbeddings: Embedding model wrapper (e.g., text-embedding-005)
# - VectorSearchVectorStore: LangChain vector store backed by Vertex AI Vector Search
# - VectorSearchVectorStoreDatastore: variant that uses Datastore for document storage
from langchain_google_vertexai import VertexAI
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_vertexai import (
    VectorSearchVectorStore,
    VectorSearchVectorStoreDatastore,
)

In [5]:
# ---- IMPORT OTHERS ----
import textwrap

### 1.5 Verify library versions


In [ ]:
# Check versions of the main SW platforms of the system
# GCP aiplatform & vertex ai
import vertexai
from google.cloud import aiplatform
print(f"aiplatform SDK version: {aiplatform.__version__}")
print(f"Vertex AI SDK version: {vertexai.__version__}")

# LangChain, langchain-core, langchain-community
import langchain
print(f"LangChain version: {langchain.__version__}")

from langchain_core import __version__ as langchain_core_version
print(f"langchain-core version: {langchain_core_version}")

from langchain_classic import __version__ as langchain_classic_version
print(f"langchain-classic version: {langchain_classic_version}")

from langchain_community import __version__ as langchain_community_version
print(f"langchain-community version: {langchain_community_version}")

from langchain_google_community import __version__ as langchain_google_community_version
print(f"langchain-google-community version: {langchain_google_community_version}")

### 1.6 Build system development environment (paths + region)


In [6]:
# BUILD SYSTEM DEVELOPMENT ENVIRONMENT
# Already set PROJECT_ID: PROJECT_ID = 'precise-machine-471801-n5'

REGION = "us-central1"
BUCKET_NAME = "adta5770-docs-folder-group4-kp"   # Group 4's bucket (created during HW4)
folder_prefix = "documents/pdfs/"

BUCKET_URI = f"gs://{BUCKET_NAME}/{folder_prefix}"

print(f"BUCKET_URI: {BUCKET_URI}")

BUCKET_URI: gs://adta5770-docs-folder-group4-kp/documents/pdfs/


In [ ]:
# ALL PDFS ARE ALREADY IN THE BUCKET - DON'T DO ANYTHING HERE
""" COMMENT ALL
!gcloud storage cp -r gs://github-repo/documents/google-research-pdfs/* {BUCKET_URI}
"""

# Verify the PDFs are visible in the bucket
!gsutil ls {BUCKET_URI} | head -5
!echo "Total PDF count:"
!gsutil ls {BUCKET_URI} | wc -l

### 1.7 Initialize GCP AI Platform for the AI SW system


In [7]:
# --------------------------------------------------------------------
# INITIALIZE GCP AI PLATFORM FOR AI SW SYSTEM
# --------------------------------------------------------------------
# Initialize the system (the current AI software application)
# This AI software application is associated with the GCP project as declared
# This AI software application runs at the specified GCP region
# This AI software application uses the specified GCS bucket
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
print("GCP AI Platform initialized.")

GCP AI Platform initialized.


### 1.8 Define text embedding model & index constants


In [8]:
# --------------------------------------------------------------------
# DEFINE TEXT EMBEDDING MODEL CREDENTIALS & CREATE IT
# --------------------------------------------------------------------

# The number of dimensions for text-embedding-005 is 768.
# The text-embedding-005 model is Google's latest text embedding model,
# which replaced the older gecko models. It produces 768-dimensional vectors,
# same as gecko@003.
DIMENSIONS = 768

# Index Constants
DISPLAY_NAME_ID = "adta5770_group4_index"       # Display name for the Matching Engine index
DEPLOYED_INDEX_ID = "adta5770_group4_endpoint"  # Deployed index / endpoint ID

# Create text embedding model based on GCP Vertex AI: text-embedding-005
embedding_model = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project=PROJECT_ID,
)
print(f"Embedding model ready: text-embedding-005 ({DIMENSIONS} dims)")

/tmp/ipykernel_4216/4066286831.py:16: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding_model = VertexAIEmbeddings(
/tmp/ipykernel_4216/4066286831.py:16: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embedding_model = VertexAIEmbeddings(


Embedding model ready: text-embedding-005 (768 dims)


### 1.9 Check for existing indexes & endpoints (clean slate verification)


In [ ]:
# First make sure the Python variable exists
PROJECT_ID = 'precise-machine-471801-n5'

# Enable required Google Cloud APIs — use {PROJECT_ID} not $PROJECT_ID
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}
!gcloud services enable storage.googleapis.com --project={PROJECT_ID}
!gcloud services enable generativelanguage.googleapis.com --project={PROJECT_ID}
!gcloud services enable cloudresourcemanager.googleapis.com --project={PROJECT_ID}

# Verify
!gcloud services list --enabled --project={PROJECT_ID} | grep -E "aiplatform|storage|generativelanguage"

# Wait for propagation
import time
print("Waiting 60 seconds for API activation to propagate...")
time.sleep(60)
print("Ready. Re-run the 'CHECK TO FIND ALL EXISTING INDICES' cell now.")

In [ ]:
# CHECK TO FIND ALL EXISTING INDICES AND ENDPOINTS
list_indexes = aiplatform.MatchingEngineIndex.list()
print("Existing indexes:")
print(list_indexes)
print("\n\n")

list_end_points = aiplatform.MatchingEngineIndexEndpoint.list()
print("Existing endpoints:")
print(list_end_points)

## PHASE 2: PROCESS DOCUMENTS
Make the PDF corpus ready for embedding and vectorization.


### 2.1 List blobs (PDFs) in the GCS bucket


In [ ]:
# For GCS: Google Cloud Storage access and loading
# In GCS: Each document/file is a 'blob' (SQL: Blob is a collection of text)
# Knowledge base: named as 'documents_from_blob'
# Knowledge base is a storage/data structure to store the documents

from google.cloud import storage
client = storage.Client()

for blob in client.list_blobs(BUCKET_NAME, prefix=folder_prefix):
    print(str(blob))

<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/A Literature Analysis of Walmart's Supply Chain Excellence in terms of Integration, Distribution and Operations.pdf, 1775554210149216>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/A Review of Supply Chain Risk Management - Definition, Theory, and Research Agenda.pdf, 1775554214850543>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/Agility and Resilience in Supply Chains Enhancing Financial Performance.pdf, 1775554218934614>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/Artificial intelligence in supply chain management - A systematic literature.pdf, 1775554225781250>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/Asymmetric Inventory Management.pdf, 1775554218857044>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/Building a Resilient Supply Chain - Strategic Guide for Businesses.pdf, 1775554227866021>
<Blob: adta5770-docs-folder-group4-kp, documents/pdfs/Comprehensive Supply Chain Risk Managemen

### 2.2 Load PDF files in the GCS bucket into a KNOWLEDGE BASE
NOTE: It can take 5 minutes or more.


In [ ]:
import os
import tempfile
from langchain_community.document_loaders import PyPDFLoader

print(f"Processing documents from {BUCKET_URI}")
bucket = storage_client.bucket(BUCKET_NAME)

all_documents = []
blobs = list(bucket.list_blobs(prefix=folder_prefix))
pdf_blobs = [b for b in blobs if not b.name.endswith("/") and b.name.lower().endswith(".pdf")]
print(f"Found {len(pdf_blobs)} PDFs")

with tempfile.TemporaryDirectory() as tmp_dir:
    for i, blob in enumerate(pdf_blobs, 1):
        name = blob.name.split("/")[-1]
        print(f"  [{i:3d}/{len(pdf_blobs)}] {name[:80]}")

        # Download blob to local temp file
        local_path = os.path.join(tmp_dir, f"doc_{i}.pdf")
        blob.download_to_filename(local_path)

        # Fast text extraction with PyPDFLoader (no OCR, no layout analysis)
        loader = PyPDFLoader(local_path)
        try:
            docs = loader.load()
        except Exception as e:
            print(f"      SKIP ({e.__class__.__name__}): {str(e)[:60]}")
            continue

        # Enrich metadata
        source = f"gs://{BUCKET_NAME}/" + "/".join(blob.name.split("/")[0:-1])
        for d in docs:
            d.metadata["source"] = source
            d.metadata["document_name"] = name

        all_documents.extend(docs)

print(f"\n# of document pages loaded (pre-chunking) = {len(all_documents)}")

Processing documents from gs://adta5770-docs-folder-group4-kp/documents/pdfs/
Found 100 PDFs
  [  1/100] A Literature Analysis of Walmart's Supply Chain Excellence in terms of Integrati
  [  2/100] A Review of Supply Chain Risk Management - Definition, Theory, and Research Agen
  [  3/100] Agility and Resilience in Supply Chains Enhancing Financial Performance.pdf
  [  4/100] Artificial intelligence in supply chain management - A systematic literature.pdf
  [  5/100] Asymmetric Inventory Management.pdf
  [  6/100] Building a Resilient Supply Chain - Strategic Guide for Businesses.pdf
  [  7/100] Comprehensive Supply Chain Risk Management Strategy.pdf
  [  8/100] Contemporary Challenges in Logistics and Supply Chain Management .pdf
  [  9/100] Contemporary Challenges in Logistics and Supply Chain Management.pdf
  [ 10/100] Contingency Management and Supply Chain Performance COVID-19 Pandemic.pdf
  [ 11/100] Design-Based Supply Chain Operations Research Model - Fostering Resilience And S

  [ 18/100] Enhancing supply chain management with deep learning and machine learning techni
  [ 19/100] Evaluation of Current Technology in Supply Chain Management Systems.pdf
  [ 20/100] Four Fundamentals of Supply Chain Management.pdf
  [ 21/100] Fundamentals of Supply Chain Management - Core Concepts and Principles.pdf
  [ 22/100] Global Procurement Manual -  Best Practices and Standards.pdf
  [ 23/100] Global Purchasing and Supply Management.pdf
  [ 24/100] Global Sourcing And Procurement Strategy Framework and Implementation.pdf
  [ 25/100] Global Sourcing in Procurement Management- International Strategies .pdf
  [ 26/100] Global Supply Chain Management - Automotive Industry.pdf
  [ 27/100] Global Supply Chain Strategy.pdf
  [ 28/100] Green Logistics Practices and Their Impact on Environmental Sustainability.pdf
  [ 29/100] Impact of Perceived Benefits on Blockchain Adoption in SCM.pdf
  [ 30/100] Impact of inventory management.pdf
  [ 31/100] Incentives in Inventory Management.

  [ 35/100] International Procurement Module 1 Basics and Fundamentals.pdf
  [ 36/100] International Procurement Practices and Supply Chain Performance.pdf
  [ 37/100] Inventory Management Optimization.pdf
  [ 38/100] Inventory Management Strategies to Enhance Resilience in Automotive Industry.pdf
  [ 39/100] Inventory Management as a Key Driver of Sustainability.pdf
  [ 40/100] Inventory Optimization and Demand Forecasting - Balancing Stock Levels and Servi
  [ 41/100] Inventory management optimization  a case study of Amazon supply chain.pdf.pdf
  [ 42/100] Issues in Supply Chain Management Contemporary Challenges.pdf
  [ 43/100] Large Language Models for Supply Chain Optimization.pdf
  [ 44/100] Logistics Warehouse Optimization- Proposal and Implementation .pdf
  [ 45/100] Logistics and Supply Chain Management (4th Edition Overview).pdf
  [ 46/100] Logistics and Supply Chain Management Development- Past, Present and Future .pdf
  [ 47/100] Logistics and Warehousing Management.pdf
  